# Microbial General Model 2 · Getting started

From a microbial abundance table to community representations, contextual OTU states and sparse features.

**What you will run:** Small inference, a CSV/CLI round trip, and the formal XLarge layer-10 SAE. This tutorial uses constructed abundances over eight real OTU vocabulary entries to demonstrate the interface. These communities are examples, not experimental observations.

Install `.[notebook]` from the source checkout, select the **Python (MGM2)** kernel, and set `MODEL_ROOT` to the matching model bundle. See [Installation](../docs/INSTALLATION.md). The default device is CPU; set `MGM2_DEVICE=cuda` to use a compatible GPU.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from mgm2 import MGM2, EmbeddingBank, prepare_sample, collate_samples, load_sae, __version__

torch.set_num_threads(min(4, torch.get_num_threads()))
device = os.environ.get("MGM2_DEVICE", "cpu")
if os.environ.get("MODEL_ROOT"):
    root = Path(os.environ["MODEL_ROOT"]).expanduser().resolve()
else:
    candidates = [p / "huggingface" for p in [Path.cwd(), *Path.cwd().parents]]
    root = next((p for p in candidates if (p / "models/small/config.json").is_file()), None)
    if root is None:
        raise FileNotFoundError("Set MODEL_ROOT to the model bundle before running this notebook.")
required = ["models/small/model.safetensors", "models/xlarge/model.safetensors",
            "embeddings/otus_99_embeddings.h5", "sae/otu_l10_k64/model.safetensors"]
missing = [name for name in required if not (root / name).is_file()]
if missing:
    raise FileNotFoundError(f"Model bundle is missing: {missing}")
print(f"MGM2 {__version__} | PyTorch {torch.__version__} | device={device}")

## 1 · Prepare communities

Rows represent OTUs and columns represent communities. The API normalizes each complete sample before selecting up to 768 positive OTUs. Positive abundances below `1e-4` are floored to `1e-4`; five per-community quantile bins are indexed **0–4**. Zero counts are omitted from the token sequence.

In [ ]:
with EmbeddingBank(root / "embeddings/otus_99_embeddings.h5") as bank:
    otu_ids = bank.ids[:8]
    counts = pd.DataFrame({
        "example_A": [40, 25, 15, 10, 5, 3, 1, 1],
        "example_B": [5, 5, 10, 30, 25, 15, 8, 2],
        "example_C": [60, 20, 10, 5, 3, 1, 1, 0],
        "example_D": [10, 10, 20, 20, 15, 15, 5, 5],
    }, index=pd.Index(otu_ids, name="otu_id"))
    samples = [prepare_sample(counts.index, counts[name].to_numpy(), bank)
               for name in counts.columns]
display(counts)
display(pd.DataFrame({
    "sample": counts.columns,
    "observed_otus": [len(s["feature_ids"]) for s in samples],
    "sequence_matches": [s["resolution"].count("ntv3") for s in samples],
}))

## 2 · Community and contextual representations

`cls` is one vector per community; `hidden` includes CLS followed by contextual OTUs; `mean_pool` averages valid OTU states; `qwen` is the normalized semantic projection. Padding is excluded from token-level summaries. The loader enters evaluation mode.

In [ ]:
model = MGM2.from_pretrained(root / "models/small", device=device)
assert not model.training
batch = collate_samples(samples, device=device)
with torch.inference_mode():
    outputs = model(batch)
display(pd.DataFrame([{"representation": key, "shape": tuple(value.shape)}
                      for key, value in outputs.items()]))
assert outputs["cls"].shape == (4, 224)
assert torch.isfinite(outputs["cls"]).all()
valid = (batch["input_ids"] >= 0) & batch["attention_mask"].bool()
otu_states = outputs["hidden"][valid]
print("Contextual OTU states:", tuple(otu_states.shape))

In [ ]:
relative = counts.div(counts.sum(axis=0), axis=1)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.3), constrained_layout=True)
relative.T.plot.bar(stacked=True, ax=axes[0], legend=False, colormap="viridis", width=0.75)
axes[0].set(ylabel="Relative abundance", xlabel="", title="Example community inputs")
axes[0].tick_params(axis="x", rotation=0)
cls = torch.nn.functional.normalize(outputs["cls"], dim=-1).cpu().numpy()
similarity = cls @ cls.T
im = axes[1].imshow(similarity, cmap="Blues", vmin=0, vmax=1)
axes[1].set(xticks=range(4), yticks=range(4),
            xticklabels=["A", "B", "C", "D"], yticklabels=["A", "B", "C", "D"],
            title="CLS cosine similarity")
fig.colorbar(im, ax=axes[1], label="Cosine similarity")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 3 · Export a table through the CLI

The same preprocessing is available through `mgm2-embed`. Exports include sample identifiers, model identity and per-sample feature-resolution counts. Each execution creates a new output folder.

In [ ]:
import subprocess

output_parent = Path(os.environ.get("MGM2_OUTPUT_DIR", "outputs")).expanduser()
output_parent.mkdir(parents=True, exist_ok=True)
output_dir = Path(tempfile.mkdtemp(prefix="mgm2-tutorial-", dir=output_parent))
table_path, archive_path = output_dir / "example_counts.csv", output_dir / "community_embeddings.npz"
counts.to_csv(table_path)
command = [
    sys.executable, "-m", "mgm2.cli",
    "--model", str(root / "models/small"),
    "--table", str(table_path),
    "--embeddings", str(root / "embeddings/otus_99_embeddings.h5"),
    "--output", str(archive_path),
    "--device", device, "--batch-size", "2",
]
completed = subprocess.run(command, check=True, text=True, capture_output=True,
                           env={**os.environ, "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2"})
with np.load(archive_path, allow_pickle=False) as exported:
    np.testing.assert_array_equal(exported["sample_ids"], counts.columns.to_numpy())
    np.testing.assert_allclose(exported["cls"], outputs["cls"].cpu().numpy(), rtol=2e-4, atol=2e-5)
    display(pd.DataFrame(exported["resolution_counts"],
                         columns=exported["resolution_columns"], index=exported["sample_ids"]))
print("Saved the example table and verified CLI embeddings.")

## 4 · Sparse features with the matched XLarge SAE

The formal SAE uses **MGM2-XLarge layer 10**, with 4,096 dictionary features and a calibrated inference threshold. Select valid OTUs, including bin 0; exclude CLS and padding. The inference activation count can differ from the average training budget of 64.

In [ ]:
backbone = MGM2.from_pretrained(root / "models/xlarge", device=device)
sae = load_sae(root / "sae/otu_l10_k64", device=device)
with torch.inference_mode():
    states = backbone(batch)["hidden"][valid]
    features = sae.encode(states)
    reconstruction = sae.denormalize(sae.decode(features))
assert features.shape == (int(valid.sum()), 4096)
assert torch.isfinite(features).all()
activity = features.cpu().numpy()
display(pd.DataFrame({
    "OTU states": [len(states)],
    "Dictionary width": [activity.shape[1]],
    "Mean active features / OTU": [(activity > 0).sum(axis=1).mean()],
    "Reconstruction MSE (state units)": [(reconstruction - states).square().mean().item()],
}))

In [ ]:
mean_activity = activity.mean(axis=0)
top = np.argsort(mean_activity)[-12:]
fig, ax = plt.subplots(figsize=(6.2, 3.5), constrained_layout=True)
ax.barh([f"f{i}" for i in top], mean_activity[top], color="#478D91")
ax.set(xlabel="Mean SAE activation across example OTUs", title="Example sparse features")
ax.spines[["top", "right"]].set_visible(False)
plt.show()
np.savez_compressed(output_dir / "sparse_features.npz",
                    features=activity,
                    sample_index=valid.nonzero(as_tuple=True)[0].cpu().numpy(),
                    otu_ids=np.asarray([otu for s in samples for otu in s["feature_ids"]], dtype=str))
print("Saved sparse features with OTU and sample mapping.")

## Next steps

Replace the constructed table with your feature-by-sample counts and the matching sequence embedding bank. Taxonomic inputs can use the optional fallback bank; inspect the exported resolution counts alongside the embeddings.

- [Model and preprocessing contract](../docs/MODEL.md)
- [CLI and representation guide](../README.md#usage)
- [Aggregate article benchmarks](../benchmarks/README.md)
- [SAE interpretation agent](https://github.com/LudensZhang/mgm2-sae-agent)

The plots above describe the demonstration inputs. Biological interpretation uses the feature atlas and the agent's quantitative evidence and source records.